# ML Model — Patient Deterioration Predictor
**Medallion Healthcare Analytics Platform**  
Celebal Excellence Internship 2025

> Trains a Random Forest classifier on 417,866 patient records to predict whether a patient will clinically deteriorate within the next 12 hours. AUC-ROC: **0.9442**

## Problem Statement

| Aspect | Detail |
|--------|--------|
| **Target** | `deterioration_next_12h` (binary: 0/1) |
| **Class imbalance** | 95% No Deterioration / 5% Deterioration |
| **Features** | 18 clinical + 12 engineered = 30 total |
| **Algorithm** | Random Forest (class_weight='balanced') |
| **Validation** | 5-fold stratified cross-validation |
| **Key metric** | AUC-ROC (handles class imbalance well) |

## 1. Setup & Data Loading

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
import json
from config.settings import RAW_DATA_DIR, MODEL_DIR, ML_FEATURES, ML_TARGET

In [ ]:
from ml.train_model import load_training_data
df = load_training_data()
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df[ML_TARGET].value_counts())
print(f'Positive rate: {df[ML_TARGET].mean():.2%}')

## 2. Exploratory Data Analysis

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Vital sign distributions by class
fig, axes = plt.subplots(2, 3, figsize=(14,8))
vitals_to_plot = ['heart_rate','spo2_pct','respiratory_rate',
                   'systolic_bp','temperature_c','sepsis_risk_score']

for ax, col in zip(axes.flat, vitals_to_plot):
    for label, group in df.groupby(ML_TARGET)[col]:
        ax.hist(group.dropna(), bins=40, alpha=0.6,
                label='Deterioration' if label==1 else 'Normal',
                color='#ef4444' if label==1 else '#3b82f6')
    ax.set_title(col.replace('_',' ').title(), fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

fig.suptitle('Vital Distributions: Normal vs Deterioration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/vital_distributions.png', dpi=120, bbox_inches='tight',
            facecolor='white')
print('Plot saved to notebooks/vital_distributions.png')
plt.close()

In [ ]:
# Key statistics comparison
deterioration = df[df[ML_TARGET]==1]
normal        = df[df[ML_TARGET]==0]
stats = pd.DataFrame({
    'Normal (avg)':       normal[ML_FEATURES].mean(),
    'Deterioration (avg)': deterioration[ML_FEATURES].mean(),
})
stats['Difference'] = stats['Deterioration (avg)'] - stats['Normal (avg)']
stats['Pct Change'] = (stats['Difference'] / stats['Normal (avg)'].abs() * 100).round(1)
stats.sort_values('Difference', key=abs, ascending=False).round(3)

## 3. Feature Engineering

We add 12 clinical interaction features on top of the 18 base features:

In [ ]:
from ml.train_model import engineer_features, get_feature_columns
df_eng = engineer_features(df)
feat_cols = get_feature_columns(df_eng)
print(f'Base features: {len(ML_FEATURES)}')
print(f'Engineered features added: {len(feat_cols)-len(ML_FEATURES)}')
print(f'Total features: {len(feat_cols)}')
print('\nEngineered features:')
for c in feat_cols:
    if c not in ML_FEATURES:
        print(f'  + {c}')

## 4. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_eng[feat_cols].fillna(0).values
y = df[ML_TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')
print(f'Features:     {X_train.shape[1]}')
print(f'Positive rate (train): {y_train.mean():.2%}')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), cw))
print('Class weights:', {int(k):round(v,3) for k,v in class_weights.items()})

rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', n_jobs=-1, random_state=42
)
rf.fit(X_train_s, y_train)
print('Training complete!')

## 5. Model Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, f1_score, confusion_matrix
)

y_pred = rf.predict(X_test_s)
y_prob = rf.predict_proba(X_test_s)[:,1]

auc_roc  = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)
f1       = f1_score(y_test, y_pred)

print(f'AUC-ROC          : {auc_roc:.4f}')
print(f'Avg Precision    : {avg_prec:.4f}')
print(f'F1 Score         : {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Deterioration','Deterioration']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print('Confusion Matrix:')
print(f'  True Negatives  (correct no-risk): {tn:>6,}')
print(f'  False Positives (false alarm):     {fp:>6,}')
print(f'  False Negatives (missed cases):   {fn:>6,}  ← Clinical priority to minimise')
print(f'  True Positives  (correct risk):   {tp:>6,}')

## 6. Feature Importance

In [ ]:
fi_df = pd.read_csv(os.path.join(MODEL_DIR,'feature_importance.csv'))
print('Top 15 most important features:')
print(fi_df.head(15).to_string(index=False))

## 7. Load Saved Model & Score New Patient

In [ ]:
from ml.predict import Predictor
p = Predictor()
print('Model info:')
for k,v in p.get_model_info().items():
    print(f'  {k:<20}: {v}')

In [ ]:
# Score a new high-risk patient
high_risk_patient = {
    'heart_rate': 142, 'spo2_pct': 84, 'respiratory_rate': 29,
    'systolic_bp': 82, 'diastolic_bp': 50, 'temperature_c': 39.3,
    'oxygen_flow': 8, 'mobility_score': 1, 'nurse_alert': 1,
    'wbc_count': 19.5, 'lactate': 3.8, 'creatinine': 2.4,
    'crp_level': 135, 'hemoglobin': 8.5, 'sepsis_risk_score': 0.89,
    'age': 74, 'comorbidity_index': 6, 'hour_from_admission': 10,
}
result = p.score_single(high_risk_patient)
print(f'Risk Score : {result["risk_score"]:.4f}')
print(f'Risk Band  : {result["risk_band"]}')
print(f'Alert      : {result["alert"]}')
print(f'Reasons    : {result["explanation"]}')

In [ ]:
# Score a low-risk patient
low_risk_patient = {
    'heart_rate': 72, 'spo2_pct': 98, 'respiratory_rate': 14,
    'systolic_bp': 128, 'diastolic_bp': 82, 'temperature_c': 36.8,
    'oxygen_flow': 0, 'mobility_score': 4, 'nurse_alert': 0,
    'wbc_count': 7.2, 'lactate': 0.9, 'creatinine': 0.8,
    'crp_level': 5, 'hemoglobin': 13.5, 'sepsis_risk_score': 0.08,
    'age': 45, 'comorbidity_index': 1, 'hour_from_admission': 3,
}
result2 = p.score_single(low_risk_patient)
print(f'Risk Score : {result2["risk_score"]:.4f}')
print(f'Risk Band  : {result2["risk_band"]}')
print(f'Alert      : {result2["alert"]}')

## 8. Model Metrics Summary

| Metric | Value | Interpretation |
|--------|-------|----------------|
| **AUC-ROC** | 0.9442 | Excellent discriminative ability |
| **CV AUC-ROC** | 0.9214 ± 0.004 | Stable across 5 folds |
| **Recall** | 0.79 | 79% of real deteriorations caught |
| **Precision** | 0.36 | 36% of alerts are true positives |
| **F1** | 0.50 | Balanced harmonic mean |

**Clinical interpretation:** High recall (79%) is the priority — we catch most deteriorating patients at the cost of some false alarms. False negatives (missed deteriorations) are more dangerous than false positives (unnecessary reviews).